# Customer Churn Prediction — End-to-End ML PipelineThis notebook demonstrates a **classical machine learning pipeline** that:1. **Generates** a synthetic customer churn dataset2. **Loads** the data into the DataLake (Impala) via `cml.data_v1`3. **Ingests** training data back from the lake4. **Trains** a scikit-learn model with proper feature preprocessing5. **Predicts** churn on unseen holdout customers> **Business context:** Customer churn (attrition) is when subscribers leave a service. Predicting churn early lets teams offer retention incentives before customers cancel.

## 0. Setup & DependenciesWe need `scikit-learn` for the ML pipeline. Run this cell once per session if it is not already installed.

In [ ]:
%pip install scikit-learn -q


## 1. ImportsStandard libraries plus Cloudera ML data connection and scikit-learn building blocks.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cml.data_v1 as cmldata

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

CONNECTION_NAME = "default-impala-aws"
DB = "default"
TRAIN_TABLE = "customer_churn_training"
SCORE_TABLE = "customer_churn_scoring"
BATCH_SIZE = 200

print("Imports loaded successfully.")


## 2. Connect to the DataLakeWe use the platform-managed Impala connection. JWT authentication is handled automatically in CML Workbench.

In [ ]:
conn = cmldata.get_connection(CONNECTION_NAME)
cursor = conn.get_cursor()

databases = conn.get_pandas_dataframe("SHOW DATABASES")
print("Connected to:", CONNECTION_NAME)
print(databases.to_string(index=False))


## 3. Generate Synthetic Churn DatasetWe simulate ~5,000 telecom-style customer records. The **churn label** is generated using a logistic-style probability so that short tenure, month-to-month contracts, and high support tickets correlate with higher churn — giving the model real patterns to learn.The data is split **80/20 (stratified)** into training and scoring (unseen) sets.

In [ ]:
N_CUSTOMERS = 5000

contract_types = ["Month-to-month", "One year", "Two year"]
payment_methods = ["Electronic check", "Mailed check", "Bank transfer", "Credit card"]
internet_services = ["DSL", "Fiber optic", "None"]
yes_no = ["Yes", "No"]

customer_ids = [f"CUST_{i:05d}" for i in range(1, N_CUSTOMERS + 1)]
tenure = np.random.randint(0, 73, N_CUSTOMERS)
monthly_charges = np.round(np.random.uniform(20, 120, N_CUSTOMERS), 2)
total_charges = np.round(tenure * monthly_charges * np.random.uniform(0.85, 1.05, N_CUSTOMERS), 2)
contract = np.random.choice(contract_types, N_CUSTOMERS, p=[0.45, 0.30, 0.25])
payment = np.random.choice(payment_methods, N_CUSTOMERS)
internet = np.random.choice(internet_services, N_CUSTOMERS, p=[0.35, 0.40, 0.25])
online_security = np.where(internet == "None", "No internet service", np.random.choice(["Yes", "No"], N_CUSTOMERS))
tech_support = np.where(internet == "None", "No internet service", np.random.choice(["Yes", "No"], N_CUSTOMERS))
paperless = np.random.choice(yes_no, N_CUSTOMERS)
senior = np.random.binomial(1, 0.2, N_CUSTOMERS)
support_tickets = np.random.poisson(1.5, N_CUSTOMERS).clip(0, 8)
usage_gb = np.round(np.random.exponential(80, N_CUSTOMERS), 1)

logit = (
    -1.5
    + 0.04 * (12 - tenure)
    + 0.8 * (contract == "Month-to-month")
    + 0.35 * (payment == "Electronic check")
    + 0.25 * (internet == "Fiber optic")
    + 0.3 * (online_security == "No")
    + 0.3 * (tech_support == "No")
    + 0.25 * support_tickets
    + 0.3 * senior
    - 0.005 * usage_gb
)
prob_churn = 1 / (1 + np.exp(-logit))
churn = (np.random.random(N_CUSTOMERS) < prob_churn).astype(int)

df_all = pd.DataFrame({
    "customer_id": customer_ids,
    "tenure_months": tenure,
    "monthly_charges": monthly_charges,
    "total_charges": total_charges,
    "contract_type": contract,
    "payment_method": payment,
    "internet_service": internet,
    "online_security": online_security,
    "tech_support": tech_support,
    "paperless_billing": paperless,
    "senior_citizen": senior,
    "num_support_tickets": support_tickets,
    "avg_monthly_usage_gb": usage_gb,
    "churn": churn,
})

df_train, df_score = train_test_split(
    df_all, test_size=0.2, random_state=RANDOM_STATE, stratify=df_all["churn"]
)

print(f"Total customers : {len(df_all):,}")
print(f"Training set    : {len(df_train):,}  (churn rate: {df_train['churn'].mean():.1%})")
print(f"Scoring set     : {len(df_score):,}  (churn rate: {df_score['churn'].mean():.1%})")
df_all.head()


## 4. Load Data into ImpalaWe create two tables in the `default` database and insert rows in batches. Impala DDL/DML can take a minute or two — this is normal for lakehouse operations.> **Re-runnable:** Tables are dropped first if they already exist.

In [ ]:
def sql_str(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return "NULL"
    return "'" + str(val).replace("'", "''") + "'"


def create_churn_table(table_name):
    cursor.execute(f"DROP TABLE IF EXISTS {DB}.{table_name}")
    ddl = (
        f"CREATE TABLE {DB}.{table_name} ("
        "customer_id STRING, tenure_months INT, monthly_charges DOUBLE, "
        "total_charges DOUBLE, contract_type STRING, payment_method STRING, "
        "internet_service STRING, online_security STRING, tech_support STRING, "
        "paperless_billing STRING, senior_citizen INT, num_support_tickets INT, "
        "avg_monthly_usage_gb DOUBLE, churn INT)"
    )
    cursor.execute(ddl)
    print(f"Created {DB}.{table_name}")


def insert_batches(df, table_name, batch_size=BATCH_SIZE):
    total = len(df)
    for start in range(0, total, batch_size):
        batch = df.iloc[start : start + batch_size]
        values = []
        for _, row in batch.iterrows():
            values.append(
                f"({sql_str(row['customer_id'])}, {int(row['tenure_months'])}, "
                f"{float(row['monthly_charges'])}, {float(row['total_charges'])}, "
                f"{sql_str(row['contract_type'])}, {sql_str(row['payment_method'])}, "
                f"{sql_str(row['internet_service'])}, {sql_str(row['online_security'])}, "
                f"{sql_str(row['tech_support'])}, {sql_str(row['paperless_billing'])}, "
                f"{int(row['senior_citizen'])}, {int(row['num_support_tickets'])}, "
                f"{float(row['avg_monthly_usage_gb'])}, {int(row['churn'])})"
            )
        sql = f"INSERT INTO {DB}.{table_name} VALUES " + ", ".join(values)
        cursor.execute(sql)
        print(f"  Inserted rows {start + 1:,}–{min(start + batch_size, total):,} into {table_name}")


create_churn_table(TRAIN_TABLE)
insert_batches(df_train, TRAIN_TABLE)

create_churn_table(SCORE_TABLE)
insert_batches(df_score, SCORE_TABLE)

train_count = conn.get_pandas_dataframe(f"SELECT COUNT(*) AS cnt FROM {DB}.{TRAIN_TABLE}")
score_count = conn.get_pandas_dataframe(f"SELECT COUNT(*) AS cnt FROM {DB}.{SCORE_TABLE}")
print(f"\nVerified: {TRAIN_TABLE} = {train_count['cnt'].iloc[0]:,} rows")
print(f"Verified: {SCORE_TABLE}  = {score_count['cnt'].iloc[0]:,} rows")


## 5. Ingest Training Data from the DataLakeWe read the training table back into a pandas DataFrame — this mirrors how a real pipeline would consume curated lakehouse data.

In [ ]:
df_from_lake = conn.get_pandas_dataframe(f"SELECT * FROM {DB}.{TRAIN_TABLE}")
print(f"Loaded {len(df_from_lake):,} training rows from the DataLake")
df_from_lake.head()


## 6. Build the ML PipelineA `ColumnTransformer` applies:- **StandardScaler** to numeric features- **OneHotEncoder** to categorical featuresThese feed a **RandomForestClassifier** — a robust baseline for mixed tabular data.

In [ ]:
TARGET = "churn"
ID_COL = "customer_id"

numeric_features = [
    "tenure_months", "monthly_charges", "total_charges",
    "senior_citizen", "num_support_tickets", "avg_monthly_usage_gb",
]
categorical_features = [
    "contract_type", "payment_method", "internet_service",
    "online_security", "tech_support", "paperless_billing",
]

X = df_from_lake.drop(columns=[TARGET, ID_COL])
y = df_from_lake[TARGET]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100, random_state=RANDOM_STATE, class_weight="balanced"
    )),
])

print("Pipeline defined:", model)


## 7. Train & ValidateWe hold out 20% of the **training table** for in-sample validation, then report standard classification metrics.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

model.fit(X_tr, y_tr)
y_val_pred = model.predict(X_val)
y_val_proba = model.predict_proba(X_val)[:, 1]

print("=== Validation Metrics (from training table) ===")
print(f"Accuracy  : {accuracy_score(y_val, y_val_pred):.3f}")
print(f"Precision : {precision_score(y_val, y_val_pred):.3f}")
print(f"Recall    : {recall_score(y_val, y_val_pred):.3f}")
print(f"F1        : {f1_score(y_val, y_val_pred):.3f}")
print(f"ROC-AUC   : {roc_auc_score(y_val, y_val_proba):.3f}")
print()
print(classification_report(y_val, y_val_pred, target_names=["Stayed", "Churned"]))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_val, y_val_pred)).plot(ax=ax, cmap="Blues")
ax.set_title("Validation Confusion Matrix")
plt.tight_layout()
plt.show()


## 8. Predict Churn on Unseen DataThe **scoring table** contains customers the model has **never seen during training**. We load it from the lake, predict churn, and evaluate against the held-out labels.

In [ ]:
df_unseen = conn.get_pandas_dataframe(f"SELECT * FROM {DB}.{SCORE_TABLE}")
print(f"Loaded {len(df_unseen):,} unseen scoring rows")

X_unseen = df_unseen.drop(columns=[TARGET, ID_COL])
y_unseen = df_unseen[TARGET]

y_unseen_pred = model.predict(X_unseen)
y_unseen_proba = model.predict_proba(X_unseen)[:, 1]

results = pd.DataFrame({
    "customer_id": df_unseen[ID_COL],
    "actual_churn": y_unseen.values,
    "predicted_churn": y_unseen_pred,
    "churn_probability": np.round(y_unseen_proba, 3),
})
results["risk_tier"] = pd.cut(
    results["churn_probability"],
    bins=[0, 0.3, 0.6, 1.0],
    labels=["Low", "Medium", "High"],
)

print("\n=== Holdout Metrics (unseen scoring table) ===")
print(f"Accuracy  : {accuracy_score(y_unseen, y_unseen_pred):.3f}")
print(f"Precision : {precision_score(y_unseen, y_unseen_pred):.3f}")
print(f"Recall    : {recall_score(y_unseen, y_unseen_pred):.3f}")
print(f"F1        : {f1_score(y_unseen, y_unseen_pred):.3f}")
print(f"ROC-AUC   : {roc_auc_score(y_unseen, y_unseen_proba):.3f}")

print("\nSample predictions (first 10 rows):")
results.head(10)


In [ ]:
high_risk = results[results["risk_tier"] == "High"].sort_values("churn_probability", ascending=False)
print(f"High-risk customers flagged: {len(high_risk):,}")
high_risk.head(10)


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_unseen, y_unseen_pred)).plot(ax=ax, cmap="Oranges")
ax.set_title("Holdout (Unseen) Confusion Matrix")
plt.tight_layout()
plt.show()


## 9. Optional CleanupTables are kept by default so you can inspect them in Impala. Uncomment and run the cell below to drop them when finished.

In [ ]:
# Uncomment to remove tables:
# cursor.execute(f"DROP TABLE IF EXISTS {DB}.{TRAIN_TABLE}")
# cursor.execute(f"DROP TABLE IF EXISTS {DB}.{SCORE_TABLE}")
# print("Tables dropped.")

cursor.close()
conn.close()
print("Connection closed.")
